In [112]:
import pandas as pd
import numpy as np

In [113]:
data = pd.read_csv(
    "data/100_Portfolios_10x10_Daily_valueweighted.csv",
    parse_dates=True,
    low_memory=False
)
data = data.set_index("Date")
data.head()

,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,ME1 BM5,ME1 BM6,ME1 BM7,ME1 BM8,ME1 BM9,SMALL HiBM,...,BIG LoBM,ME10 BM2,ME10 BM3,ME10 BM4,ME10 BM5,ME10 BM6,ME10 BM7,ME10 BM8,ME10 BM9,BIG HiBM
Date,,,,,,,,,,,,,,,,,,,,,
19260701,-99.99,0.00,-99.99,1.59,-3.08,4.64,2.57,3.81,-0.52,-0.84,...,0.00,1.10,-0.15,-0.03,0.52,0.48,-0.43,-0.08,0.17,-99.99
19260702,-99.99,-0.27,-99.99,0.00,-0.97,-4.10,0.31,-0.47,2.74,-0.27,...,0.31,0.71,0.97,0.57,0.52,0.15,0.63,-0.04,0.34,-99.99
19260706,-99.99,1.01,-99.99,-4.69,2.35,-1.79,0.00,2.44,-5.18,-0.23,...,0.49,-0.19,0.89,0.31,-0.12,-0.18,-0.33,-0.40,-0.34,-99.99
19260707,-99.99,-1.67,-99.99,4.92,0.51,5.27,0.00,-0.74,-0.24,-0.02,...,-0.20,-0.01,0.66,0.38,0.03,0.25,-0.29,0.52,0.17,-99.99
19260708,-99.99,0.00,-99.99,1.56,-0.51,-1.06,0.00,4.90,0.45,0.26,...,0.56,-0.12,0.35,0.55,-0.14,0.19,-0.12,0.54,0.51,-99.99


In [114]:
data= data.replace(-99.99, np.nan)
data = data.replace(-999, np.nan)

In [115]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26110 entries, 19260701 to 20251031
Data columns (total 100 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SMALL LoBM  23562 non-null  float64
 1   ME1 BM2     23230 non-null  float64
 2   ME1 BM3     23208 non-null  float64
 3   ME1 BM4     24616 non-null  float64
 4   ME1 BM5     25171 non-null  float64
 5   ME1 BM6     25805 non-null  float64
 6   ME1 BM7     25813 non-null  float64
 7   ME1 BM8     26107 non-null  float64
 8   ME1 BM9     26110 non-null  float64
 9   SMALL HiBM  26110 non-null  float64
 10  ME2 BM1     23530 non-null  float64
 11  ME2 BM2     24323 non-null  float64
 12  ME2 BM3     25421 non-null  float64
 13  ME2 BM4     26102 non-null  float64
 14  ME2 BM5     25807 non-null  float64
 15  ME2 BM6     26110 non-null  float64
 16  ME2 BM7     26110 non-null  float64
 17  ME2 BM8     26110 non-null  float64
 18  ME2 BM9     26110 non-null  float64
 19  ME2 BM10    26110 n

In [116]:
data = data.dropna(how='any')
data = data.astype(np.float32)
data = data / 100.0

In [117]:
print(data.min().min(), data.max().max())
print(data.describe(percentiles=[0.001, 0.01, 0.99, 0.999]).T.sort_values("min").head(5))


-0.286200008392334 0.4556999969482422
              count      mean       std     min      0.1%        1%     50%  \
ME9 BM10    15754.0  0.000530  0.014778 -0.2862 -0.074816 -0.038747  0.0005   
ME2 BM10    15754.0  0.000702  0.014549 -0.2442 -0.085649 -0.037700  0.0008   
SMALL LoBM  15754.0  0.000069  0.018959 -0.2414 -0.105374 -0.055600  0.0001   
ME10 BM8    15754.0  0.000546  0.010988 -0.2327 -0.059248 -0.028100  0.0005   
ME10 BM4    15754.0  0.000478  0.010301 -0.2270 -0.053869 -0.027247  0.0005   

                 99%     99.9%     max  
ME9 BM10    0.038647  0.081569  0.1518  
ME2 BM10    0.038600  0.071925  0.4557  
SMALL LoBM  0.058800  0.125000  0.2500  
ME10 BM8    0.029400  0.052374  0.1052  
ME10 BM4    0.027447  0.048649  0.1054  


Podaci od 02.07.1945 - 31.10.2025.

In [118]:
from sklearn.model_selection import TimeSeriesSplit

max_years_lookback = 15

tscv_y = TimeSeriesSplit(n_splits=30, max_train_size=max_years_lookback*252, test_size=252)
tscv_q = TimeSeriesSplit(n_splits=20, max_train_size=max_years_lookback*252, test_size=63)
tscv_m = TimeSeriesSplit(n_splits=10, max_train_size=max_years_lookback*252, test_size=21)

tscv_init = TimeSeriesSplit(n_splits=5, max_train_size=max_years_lookback*252, test_size=252)

In [119]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [120]:
class WindowDataset(Dataset):
    def __init__(self, X_np, orig_idx, W, stride):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.orig_idx = np.asarray(orig_idx)
        self.W = W
        self.stride = stride
        self.n = (len(self.X) - W) // stride + 1

    def __len__(self):
        return self.n
    
    def __getitem__(self, idx):
        start = idx * self.stride
        end = start + self.W
        window = self.X[start:end]
        end_idx = self.orig_idx[end-1]
        return window, end_idx

In [121]:
class Encoder(nn.Module):
    def __init__(self, W, D, z_dim, hidden_dim=256):
        super().__init__()
        in_dim = W * D
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, z_dim)
        self.fc_logvar = nn.Linear(hidden_dim, z_dim)

    def forward(self, x):  # x: (B,W,D)
        h = F.relu(self.fc1(x.flatten(1)))
        return self.fc_mu(h), self.fc_logvar(h)

class DecoderGaussian(nn.Module):
    def __init__(self, W, D, z_dim, hidden_dim=256, min_logsigma=-8.0, max_logsigma=3.0):
        super().__init__()
        out_dim = W * D
        self.fc1 = nn.Linear(z_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, out_dim)
        self.fc_logsigma = nn.Linear(hidden_dim, out_dim)
        self.W, self.D = W, D
        self.min_logsigma = min_logsigma
        self.max_logsigma = max_logsigma

    def forward(self, z):
        h = F.relu(self.fc1(z))
        x_mu = self.fc_mu(h).view(-1, self.W, self.D)
        x_logsigma = self.fc_logsigma(h).view(-1, self.W, self.D)
        x_logsigma = torch.clamp(x_logsigma, self.min_logsigma, self.max_logsigma)
        return x_mu, x_logsigma

class VAE(nn.Module):
    def __init__(self, W, D, z_dim, hidden_dim=256):
        super().__init__()
        self.encoder = Encoder(W, D, z_dim, hidden_dim)
        self.decoder = DecoderGaussian(W, D, z_dim, hidden_dim)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        z_mu, z_logvar = self.encoder(x)
        z = self.reparameterize(z_mu, z_logvar)
        x_mu, x_logsigma = self.decoder(z)
        return x_mu, x_logsigma, z_mu, z_logvar


In [122]:
import math
def vae_loss_gaussian(x, x_mu, x_logsigma, z_mu, z_logvar, beta=0.2):
    # recon NLL for diagonal Gaussian: sum over (W,D), mean over batch
    # nll = 0.5 * [((x-mu)/sigma)^2 + 2*log(sigma) + log(2pi)]
    recon = 0.5 * (
        ((x - x_mu) / torch.exp(x_logsigma))**2 +
        2.0 * x_logsigma +
        math.log(2.0 * math.pi)
    )
    recon = recon.sum(dim=(1, 2)).mean()

    # KL(q(z|x) || N(0,I)) for diagonal Gaussian
    kl = -0.5 * (1.0 + z_logvar - z_mu.pow(2) - z_logvar.exp())
    kl = kl.sum(dim=1).mean()

    loss = recon + beta * kl
    return loss, recon, kl


In [123]:
Ws = [21, 63, 252]
splits = tscv_init.split(data)
train_idx, test_idx = list(splits)[-1]
X_train_np = data.iloc[train_idx].values
X_test_np = data.iloc[test_idx].values

train_mean = X_train_np.mean(axis=0, keepdims=True)
train_std  = X_train_np.std(axis=0, keepdims=True) + 1e-8

X_train_s = (X_train_np - train_mean) / train_std
X_test_s  = (X_test_np  - train_mean) / train_std

dataset_train = WindowDataset(X_train_s, train_idx, W=21, stride=1)
dataset_test = WindowDataset(X_test_s, test_idx, W=21, stride=1)

train_loader = DataLoader(dataset_train, batch_size=256, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset_test, batch_size=256, shuffle=False)

In [124]:
vae = VAE(W=21, D=X_train_np.shape[1], z_dim=10, hidden_dim=256).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

def run_epoch(loader, training: bool, beta=0.2):
    vae.train(training)
    total_loss = total_recon = total_kl = 0.0
    n_batches = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, _end_idx in loader:
            x = x.to(device)                # (B,W,D)

            x_mu, x_logsigma, z_mu, z_logvar = vae(x)
            loss, recon, kl = vae_loss_gaussian(x, x_mu, x_logsigma, z_mu, z_logvar, beta=beta)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            total_recon += recon.item()
            total_kl += kl.item()
            n_batches += 1

    return total_loss/n_batches, total_recon/n_batches, total_kl/n_batches

for epoch in range(1, 101):
    tr_loss, tr_rec, tr_kl = run_epoch(train_loader, training=True, beta=0.5)
    te_loss, te_rec, te_kl = run_epoch(test_loader,  training=False, beta=0.5)

    if epoch % 5 == 0:
        print(f"ep={epoch:03d}  train loss={tr_loss:.3f} (rec={tr_rec:.3f}, kl={tr_kl:.3f})"
              f"  test loss={te_loss:.3f} (rec={te_rec:.3f}, kl={te_kl:.3f})")

ep=005  train loss=3076.166 (rec=3048.933, kl=54.468)  test loss=3051.722 (rec=3034.446, kl=34.552)
ep=010  train loss=2857.230 (rec=2823.818, kl=66.823)  test loss=2734.187 (rec=2714.796, kl=38.780)
ep=015  train loss=2776.902 (rec=2743.252, kl=67.300)  test loss=2684.473 (rec=2665.022, kl=38.902)
ep=020  train loss=2745.805 (rec=2711.813, kl=67.985)  test loss=2670.398 (rec=2650.869, kl=39.059)
ep=025  train loss=2707.942 (rec=2674.081, kl=67.723)  test loss=2631.470 (rec=2611.846, kl=39.248)
ep=030  train loss=2675.010 (rec=2641.789, kl=66.442)  test loss=2630.553 (rec=2610.808, kl=39.489)
ep=035  train loss=2657.315 (rec=2623.057, kl=68.517)  test loss=2604.181 (rec=2584.307, kl=39.749)
ep=040  train loss=2639.267 (rec=2605.374, kl=67.785)  test loss=2600.278 (rec=2580.262, kl=40.032)
ep=045  train loss=2641.399 (rec=2606.508, kl=69.781)  test loss=2598.388 (rec=2578.227, kl=40.322)
ep=050  train loss=2613.811 (rec=2579.294, kl=69.035)  test loss=2593.206 (rec=2572.892, kl=40.627)


In [125]:
@torch.no_grad()
def sample_windows(vae, n, device, stochastic=False):
    vae.eval()
    z = torch.randn(n, vae.encoder.fc_mu.out_features, device=device)  # (n, z_dim)
    x_mu, x_logsigma = vae.decoder(z)

    if not stochastic:
        return x_mu  # (n, W, D)

    eps = torch.randn_like(x_mu)
    return x_mu + torch.exp(x_logsigma) * eps
